<a href="https://colab.research.google.com/github/jzazueta/ds-business-economics/blob/main/notebooks/ch14_deeplearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 14: Deep Learning: From Labels to Representations

**Data Science for Business and Economics: A Python Approach**  
Jorge Zazueta Gutiérrez — School of Economics, UASLP

---

**Scenario:** Mariana Garza is the assortment analyst at Mercados Zapalinamé, a 300-store supermarket chain headquartered in Saltillo. For the spring shelf reset, each store receives three products it does not carry yet, and she has to recommend which three. This notebook follows her from the sparse store-SKU sales grid to boosted trees, neural networks, embeddings, attention and the final recommendation.

> **Note:** Run the cells in order. Training is fast on Colab's standard CPU runtime; the whole notebook takes a few minutes. Printed numbers match the book when the same library versions are used; small differences in the last digit can appear with other versions of PyTorch or LightGBM.

## Setup

In [1]:
# Download the chapter's data from the companion repository
# (skipped if the files are already in the working folder)
import os, urllib.request
BASE = "https://raw.githubusercontent.com/jzazueta/ds-business-economics/main/data/"
for name in ["ch14_stores.csv", "ch14_skus.csv", "ch14_sales.csv"]:
    if not os.path.exists(name):
        urllib.request.urlretrieve(BASE + name, name)
        print("downloaded", name)

# LightGBM ships with Colab; install it elsewhere if missing
try:
    import lightgbm
except ImportError:
    %pip install -q lightgbm

# Recent LightGBM versions warn about an argument the book uses
import warnings
warnings.filterwarnings("ignore", message=".*eval_set.*")

---
## Filling In the Shelf

### Mariana's Data

**Code Example 14.1** — Loading the three tables.

In [2]:
import numpy as np
import pandas as pd

stores = pd.read_csv("ch14_stores.csv")   # one row per store
skus = pd.read_csv("ch14_skus.csv")       # one row per SKU
sales = pd.read_csv("ch14_sales.csv")     # one row per pair

# Number of rows and columns in each table
for name, table in [("stores", stores), ("skus", skus),
                    ("sales", sales)]:
    print(name, table.shape)
print(stores.head(3))
print(sales.head(3))

stores (300, 6)
skus (400, 5)
sales (31695, 4)
  store_id       city       state format  floor_m2 nse
0     S000  Zacatecas   Zacatecas  super      1290   C
1     S001  Monterrey  Nuevo León  super      1690  C+
2     S002  Monterrey  Nuevo León  super      2020   C
  store_id sku_id  units_13wk  units_per_week
0     S000   P003          17            1.31
1     S000   P004          11            0.85
2     S000   P005          33            2.54


**Code Example 14.2** — Where the stores are and how large they are.

In [3]:
print(stores["city"].value_counts())
print(stores["format"].value_counts())

city
Saltillo          70
Monterrey         60
Torreón           45
Nuevo Laredo      25
Monclova          22
Gómez Palacio     20
Piedras Negras    18
Zacatecas         16
Acuña             12
Ramos Arizpe      12
Name: count, dtype: int64
format
super      144
express    119
hiper       37
Name: count, dtype: int64


### A Grid with Most Cells Empty

**Code Example 14.3** — How much of the store-SKU grid is observed.

In [4]:
possible = len(stores) * len(skus)
print("possible pairs:", possible)
print("carried pairs: ", len(sales))
print("share carried:  %.3f" % (len(sales) / possible))

# SKUs carried per store, summarized by format
carried = (sales.groupby("store_id").size()
           .rename("skus_carried"))
print(stores.join(carried, on="store_id")
      .groupby("format")["skus_carried"]
      .agg(["mean", "min", "max"]).round(1))

possible pairs: 120000
carried pairs:  31695
share carried:  0.264
          mean  min  max
format                  
express   59.3   43   75
hiper    199.6  177  222
super    119.8   95  141


**Code Example 14.4** — Weekly sales per carried pair.

In [5]:
print(sales["units_per_week"].describe().round(2))

count    31695.00
mean         9.60
std         19.81
min          0.00
25%          2.38
50%          5.08
75%         10.54
max        949.00
Name: units_per_week, dtype: float64


### What the Store Columns Do Not Say

**Code Example 14.5** — Brand tier sales in border and interior stores.

In [6]:
# One frame with store and SKU attributes on every carried pair
pairs = (sales.merge(stores, on="store_id")
              .merge(skus, on="sku_id"))
pairs["log_units"] = np.log1p(pairs["units_per_week"])

border_cities = ["Piedras Negras", "Acuña", "Nuevo Laredo"]
pairs["border"] = pairs["city"].isin(border_cities)

# Mean log sales by brand tier, border vs. interior stores
table = pairs.pivot_table(index="brand_tier", columns="border",
                          values="log_units", aggfunc="mean")
table.columns = ["interior", "border"]
table["gap"] = table["border"] - table["interior"]
print(table.round(3))

               interior  border    gap
brand_tier                            
imported          1.716   2.209  0.493
national          1.882   1.837 -0.044
private_label     1.942   2.014  0.073


### Why Identity is Not Enough

---
## Boosted Trees

### Holding Pairs Back

**Code Example 14.6** — Splitting the carried pairs into training, validation and test sets.

In [7]:
# Text columns become pandas categories, which LightGBM
# and the embedding layers both need
cat_cols = ["store_id", "sku_id", "city", "state", "format",
            "nse", "category", "brand_tier", "pack"]
for col in cat_cols:
    pairs[col] = pairs[col].astype("category")
pairs["log_price"] = np.log(pairs["price_mxn"])

# Shuffle the rows once, then cut 70/10/20
rng = np.random.default_rng(14)
order = rng.permutation(len(pairs))
n_train = int(0.7 * len(pairs))
n_valid = int(0.1 * len(pairs))
train = pairs.iloc[order[:n_train]]
valid = pairs.iloc[order[n_train:n_train + n_valid]]
test = pairs.iloc[order[n_train + n_valid:]]
print(len(train), len(valid), len(test))

# Every test pair's store and SKU should appear in training
print("test rows with a new store:",
      (~test["store_id"].isin(train["store_id"])).sum())
print("test rows with a new SKU:  ",
      (~test["sku_id"].isin(train["sku_id"])).sum())

22186 3169 6340
test rows with a new store: 0
test rows with a new SKU:   0


**Code Example 14.7** — Two baselines: the mean and a level for every store and SKU.

In [8]:
import statsmodels.formula.api as smf

def rmse(actual, predicted):
    # Root mean squared error, in log units
    return np.sqrt(np.mean((actual - predicted) ** 2))

actual = test["log_units"]       # observed, test pairs
print("predict the mean:   %.3f"
      % rmse(actual, train["log_units"].mean()))

# One level per store and one per SKU, added together
levels = smf.ols("log_units ~ C(store_id) + C(sku_id)",
                 data=train).fit()
print("coefficients:", len(levels.params))
print("store + SKU levels: %.3f"
      % rmse(actual, levels.predict(test)))

predict the mean:   0.905


coefficients: 699
store + SKU levels: 0.697


### From One Tree to Many

**Code Example 14.8** — Gradient boosting written out by hand.

In [9]:
from sklearn.tree import DecisionTreeRegressor

attributes = ["city", "format", "floor_m2", "nse", "category",
              "brand_tier", "pack", "log_price"]
# Trees in scikit-learn need numbers: one column per category
X_train = pd.get_dummies(train[attributes], dtype=float)
X_valid = pd.get_dummies(valid[attributes], dtype=float)
y_train = train["log_units"].to_numpy()
y_valid = valid["log_units"].to_numpy()

RATE = 0.1                       # share of each tree to keep
# Start every prediction at the training mean
pred_train = np.full(len(train), y_train.mean())
pred_valid = np.full(len(valid), y_train.mean())
for m in range(1, 301):
    # Fit a small tree to what the ensemble still gets wrong
    residual = y_train - pred_train
    tree = DecisionTreeRegressor(max_depth=3)
    tree.fit(X_train, residual)
    # Add a fraction of its prediction to the ensemble
    pred_train += RATE * tree.predict(X_train)
    pred_valid += RATE * tree.predict(X_valid)
    if m in (1, 10, 50, 100, 300):
        print("trees %3d   train %.3f   valid %.3f"
              % (m, rmse(y_train, pred_train),
                 rmse(y_valid, pred_valid)))

trees   1   train 0.880   valid 0.881


trees  10   train 0.844   valid 0.848


trees  50   train 0.810   valid 0.815


trees 100   train 0.785   valid 0.792


trees 300   train 0.732   valid 0.749


### LightGBM

**Code Example 14.9** — LightGBM on the store and SKU attributes.

In [10]:
import lightgbm as lgb

def fit_boosting(features, num_leaves):
    model = lgb.LGBMRegressor(n_estimators=5000,
                              learning_rate=0.03,
                              num_leaves=num_leaves, verbose=-1)
    # Stop adding trees once validation error has not
    # improved for 100 rounds; keep the best round
    stop = lgb.early_stopping(100, verbose=False)
    model.fit(train[features], train["log_units"],
              eval_set=[(valid[features], valid["log_units"])],
              callbacks=[stop])
    return model

gbm_attr = fit_boosting(attributes, num_leaves=15)
print("trees kept:", gbm_attr.best_iteration_)
pred_valid = gbm_attr.predict(valid[attributes])
pred_test = gbm_attr.predict(test[attributes])
print("valid %.3f   test %.3f"
      % (rmse(valid["log_units"], pred_valid),
         rmse(actual, pred_test)))

trees kept: 4583


valid 0.653   test 0.660


**Code Example 14.10** — Two numeric columns that nearly identify stores and SKUs.

In [11]:
# Distinct values: a column with nearly one value per store
# or SKU lets a tree pick out individual stores or SKUs
print("floor areas:", stores["floor_m2"].nunique(),
      "for", len(stores), "stores")
print("prices:     ", skus["price_mxn"].nunique(),
      "for", len(skus), "SKUs")

# The same model without the two columns
coarse = [c for c in attributes
          if c not in ("floor_m2", "log_price")]
gbm_coarse = fit_boosting(coarse, num_leaves=15)
print("without them: trees %d   test %.3f"
      % (gbm_coarse.best_iteration_,
         rmse(actual, gbm_coarse.predict(test[coarse]))))

floor areas: 144 for 300 stores
prices:      309 for 400 SKUs


without them: trees 843   test 0.821


### Store and SKU as Features

**Code Example 14.11** — Adding the store and SKU identifiers, and choosing the tree size.

In [12]:
with_ids = attributes + ["store_id", "sku_id"]
models = {}
for leaves in (15, 31, 63):
    models[leaves] = fit_boosting(with_ids, leaves)
    print("leaves %2d   trees %4d   valid %.3f"
          % (leaves, models[leaves].best_iteration_,
             rmse(valid["log_units"],
                  models[leaves].predict(valid[with_ids]))))

# Choose on validation, then report the test error once
best = min(models, key=lambda k: rmse(
    valid["log_units"], models[k].predict(valid[with_ids])))
gbm = models[best]
print("chosen: %d leaves, test %.3f"
      % (best, rmse(actual, gbm.predict(test[with_ids]))))

leaves 15   trees 3337   valid 0.542


leaves 31   trees 3106   valid 0.559


leaves 63   trees 2256   valid 0.552


chosen: 15 leaves, test 0.548


**Code Example 14.12** — How much each column contributed to the trees' splits.

In [13]:
# Total loss reduction from splits on each column, as shares
gain = pd.Series(gbm.booster_.feature_importance("gain"),
                 index=with_ids)
print((gain / gain.sum()).sort_values(ascending=False).round(3))

store_id      0.506
sku_id        0.378
category      0.052
city          0.030
brand_tier    0.009
log_price     0.008
floor_m2      0.007
pack          0.005
nse           0.005
format        0.000
dtype: float64


### What the Trees Miss

**Code Example 14.13** — Average test error of the boosted model by brand tier and location.

In [14]:
# Positive error: the model predicted too little
check = test.assign(error=actual - gbm.predict(test[with_ids]))
print(check.pivot_table(index="brand_tier", columns="border",
                        values="error", aggfunc="mean")
      .rename(columns={False: "interior", True: "border"})
      .round(3))

border         interior  border
brand_tier                     
imported         -0.024   0.148
national          0.024   0.011
private_label    -0.003   0.060


---
## From a Logistic Unit to a Network

### One Unit

### Layers

### Networks in PyTorch

**Code Example 14.14** — Turning the attributes into tensors.

In [15]:
import torch

numeric = ["floor_m2", "log_price"]
categorical = [c for c in attributes if c not in numeric]
# Scale with training means and standard deviations, so
# every input column is of similar size
center = train[numeric].mean()
scale = train[numeric].std()

def design(frame):
    # One indicator column per category, plus scaled numbers
    dummies = pd.get_dummies(frame[categorical], dtype=float)
    scaled = (frame[numeric] - center) / scale
    return pd.concat([dummies, scaled], axis=1)

def to_tensor(values):
    # PyTorch's array type, in 32-bit floating point
    return torch.tensor(np.asarray(values), dtype=torch.float32)

X_train, X_valid, X_test = (to_tensor(design(f))
                            for f in (train, valid, test))
y_train, y_valid, y_test = (to_tensor(f["log_units"])
                            for f in (train, valid, test))
print(X_train.shape, X_train.dtype)
print(list(design(train).columns[:3]), "...")

torch.Size([22186, 32]) torch.float32
['city_Acuña', 'city_Gómez Palacio', 'city_Monclova'] ...


**Code Example 14.15** — One unit, with two activations.

In [16]:
import torch.nn as nn

n_inputs = X_train.shape[1]      # 32 input columns
torch.manual_seed(14)            # same random start each run
unit = nn.Linear(n_inputs, 1)    # weighted sum plus bias
print(unit.weight.shape, unit.bias.shape)
z = unit(X_train[:3])            # first three training pairs
print(z.squeeze(1))
print(torch.sigmoid(z).squeeze(1))   # logistic output
print(torch.relu(z).squeeze(1))      # ReLU, used below

torch.Size([1, 32]) torch.Size([1])
tensor([-0.3338, -0.2732,  0.2917], grad_fn=<SqueezeBackward1>)
tensor([0.4173, 0.4321, 0.5724], grad_fn=<SqueezeBackward1>)
tensor([0.0000, 0.0000, 0.2917], grad_fn=<SqueezeBackward1>)


**Code Example 14.16** — The network of Figure 14.3.

In [17]:
torch.manual_seed(14)
network = nn.Sequential(
    nn.Linear(n_inputs, 32), nn.ReLU(),   # hidden layer 1
    nn.Linear(32, 16), nn.ReLU(),         # hidden layer 2
    nn.Linear(16, 1),                     # output: log units
)
print(network)
# Count every weight and bias the network will learn
n_params = sum(p.numel() for p in network.parameters())
print("parameters:", n_params)

Sequential(
  (0): Linear(in_features=32, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=16, bias=True)
  (3): ReLU()
  (4): Linear(in_features=16, out_features=1, bias=True)
)
parameters: 1601


**Code Example 14.17** — Predictions from the network before training.

In [18]:
with torch.no_grad():            # compute only, no learning
    pred = network(X_valid).squeeze(1)
print(pred[:3])
print("untrained valid RMSE: %.3f"
      % torch.sqrt(((pred - y_valid) ** 2).mean()))

tensor([0.0226, 0.0266, 0.0343])
untrained valid RMSE: 2.046


---
## Training a Network

### The Loss and its Slope

### Backpropagation

**Code Example 14.18** — Five steps of gradient descent on one batch.

In [19]:
loss_fn = nn.MSELoss()               # mean squared error
batch = torch.arange(256)            # the first 256 pairs
# Plain gradient descent with a learning rate of 0.05
step = torch.optim.SGD(network.parameters(), lr=0.05)
for t in range(1, 6):
    pred = network(X_train[batch]).squeeze(1)
    loss = loss_fn(pred, y_train[batch])
    step.zero_grad()                 # clear old gradients
    loss.backward()                  # compute new gradients
    step.step()                      # move weights downhill
    print("step %d   loss %.3f" % (t, loss.item()))
print(network[0].weight.grad.shape)

step 1   loss 4.268
step 2   loss 3.340
step 3   loss 2.594
step 4   loss 1.980
step 5   loss 1.500
torch.Size([32, 32])


### Minibatches, Epochs and Adam

**Code Example 14.19** — Training with minibatches, Adam and early stopping.

In [20]:
def rmse_of(model, inputs, y):
    # RMSE of a model's predictions, without tracking gradients
    with torch.no_grad():
        pred = model(*inputs).squeeze(1)
        return loss_fn(pred, y).sqrt().item()

def fit_network(model, train_in, train_y, valid_in, valid_y,
                lr=1e-3, batch_size=256, max_epochs=300,
                patience=15):
    # Adam on minibatches; keep the weights of the epoch with
    # the lowest validation RMSE
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_rmse, best_epoch, best_state = float("inf"), 0, None
    history = []
    for epoch in range(1, max_epochs + 1):
        model.train()
        order = torch.randperm(len(train_y))   # new shuffle
        for start in range(0, len(train_y), batch_size):
            rows = order[start:start + batch_size]
            pred = model(*[x[rows] for x in train_in])
            loss = loss_fn(pred.squeeze(1), train_y[rows])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        model.eval()
        v = rmse_of(model, valid_in, valid_y)
        t = rmse_of(model, train_in, train_y)
        history.append((epoch, t, v))
        if v < best_rmse:                      # new best: save
            best_rmse, best_epoch = v, epoch
            state = model.state_dict()
            best_state = {k: w.clone() for k, w in state.items()}
        elif epoch - best_epoch >= patience:   # early stopping
            break
    model.load_state_dict(best_state)
    columns = ["epoch", "train", "valid"]
    return pd.DataFrame(history, columns=columns)

def make_network(hidden, seed):
    # Hidden layers of the given widths, then one output
    torch.manual_seed(seed)
    layers, width = [], n_inputs
    for h in hidden:
        layers += [nn.Linear(width, h), nn.ReLU()]
        width = h
    layers.append(nn.Linear(width, 1))
    return nn.Sequential(*layers)

network = make_network((32, 16), seed=14)
history = fit_network(network, (X_train,), y_train,
                      (X_valid,), y_valid)
best = history.loc[history["valid"].idxmin()]
print("epochs run %d, best epoch %d"
      % (len(history), best["epoch"]))
print("train %.3f   valid %.3f   test %.3f"
      % (best["train"], best["valid"],
         rmse_of(network, (X_test,), y_test)))

epochs run 101, best epoch 86
train 0.777   valid 0.801   test 0.819


### What the Network Learned

**Code Example 14.20** — Three network sizes, each trained from three random starts.

In [21]:
for hidden in [(), (32, 16), (128, 64)]:
    for seed in (14, 15, 16):
        net = make_network(hidden, seed)
        h = fit_network(net, (X_train,), y_train,
                        (X_valid,), y_valid)
        print("hidden %-9s seed %d   valid %.3f   test %.3f"
              % (hidden, seed, h["valid"].min(),
                 rmse_of(net, (X_test,), y_test)))

hidden ()        seed 14   valid 0.842   test 0.859


hidden ()        seed 15   valid 0.842   test 0.859


hidden ()        seed 16   valid 0.842   test 0.859


hidden (32, 16)  seed 14   valid 0.801   test 0.819


hidden (32, 16)  seed 15   valid 0.801   test 0.815


hidden (32, 16)  seed 16   valid 0.798   test 0.822


hidden (128, 64) seed 14   valid 0.792   test 0.809


hidden (128, 64) seed 15   valid 0.791   test 0.817


hidden (128, 64) seed 16   valid 0.795   test 0.815


---
## Embeddings

### An Indicator Column Times a Weight is a Lookup

**Code Example 14.21** — An embedding table, and the same numbers from indicator columns.

In [22]:
def codes(frame, col):
    # Position of each row's store or SKU: 0, 1, 2, ...
    return torch.tensor(frame[col].cat.codes.to_numpy(),
                        dtype=torch.long)

store_train, store_valid, store_test = (
    codes(f, "store_id") for f in (train, valid, test))
sku_train, sku_valid, sku_test = (
    codes(f, "sku_id") for f in (train, valid, test))
n_stores = len(pairs["store_id"].cat.categories)   # 300
n_skus = len(pairs["sku_id"].cat.categories)       # 400
print(store_train[:5])
print(sku_train[:5])

torch.manual_seed(14)
table = nn.Embedding(n_stores, 4)    # 300 rows of 4 numbers
lookup = table(store_train[:5])      # rows for five stores
# The same numbers: one-hot columns times a weight matrix
one_hot = nn.functional.one_hot(store_train[:5], n_stores)
print(torch.allclose(lookup, one_hot.float() @ table.weight))
print(lookup.detach())               # the five rows

tensor([ 82, 290, 245, 270,  59])
tensor([380, 357, 298,   1, 138])
True
tensor([[ 0.2121,  1.2942,  0.3106,  0.4764],
        [-0.6848,  1.1986,  1.3725, -0.5437],
        [-2.2147, -0.8943, -0.9879,  1.7342],
        [-0.3218,  0.0559, -2.1873,  1.4567],
        [-1.2112,  1.7501,  0.7274,  0.6141]])


### A Model of Taste

**Code Example 14.22** — The taste model, trained with four embedding sizes.

In [23]:
class TasteModel(nn.Module):
    # Store level + SKU level + match of store and SKU
    def __init__(self, dim):
        super().__init__()
        self.store = nn.Embedding(n_stores, dim)  # vectors
        self.sku = nn.Embedding(n_skus, dim)
        self.store_level = nn.Embedding(n_stores, 1)
        self.sku_level = nn.Embedding(n_skus, 1)
        # Small random vectors; levels start at zero
        nn.init.normal_(self.store.weight, 0, 0.1)
        nn.init.normal_(self.sku.weight, 0, 0.1)
        nn.init.zeros_(self.store_level.weight)
        nn.init.zeros_(self.sku_level.weight)
        self.mean = nn.Parameter(y_train.mean().reshape(1))

    def forward(self, store, sku):
        # Dot product: multiply the two vectors, add up
        product = self.store(store) * self.sku(sku)
        match = product.sum(1, keepdim=True)
        return (self.mean + self.store_level(store)
                + self.sku_level(sku) + match)

train_in = (store_train, sku_train)
valid_in = (store_valid, sku_valid)
test_in = (store_test, sku_test)
results = {}
for dim in (2, 4, 8, 16):
    torch.manual_seed(14)
    model = TasteModel(dim)
    h = fit_network(model, train_in, y_train,
                    valid_in, y_valid)
    results[dim] = model
    best = h.loc[h["valid"].idxmin()]
    print("dim %2d   epochs %3d   train %.3f   valid %.3f"
          % (dim, len(h), best["train"], best["valid"]))
taste = results[4]
print("dim 4 test: %.3f" % rmse_of(taste, test_in, y_test))
n_par = sum(p.numel() for p in taste.parameters())
print("parameters:", n_par)

dim  2   epochs 141   train 0.498   valid 0.568


dim  4   epochs 136   train 0.351   valid 0.420


dim  8   epochs  89   train 0.322   valid 0.456


dim 16   epochs  58   train 0.290   valid 0.488
dim 4 test: 0.423
parameters: 3501


### Structure Before Depth

**Code Example 14.23** — Embeddings followed by hidden layers instead of a dot product.

In [24]:
class DeepTaste(nn.Module):
    # Store and SKU vectors side by side, then hidden layers
    def __init__(self, dim):
        super().__init__()
        self.store = nn.Embedding(n_stores, dim)
        self.sku = nn.Embedding(n_skus, dim)
        self.layers = nn.Sequential(
            nn.Linear(2 * dim, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1))

    def forward(self, store, sku):
        both = torch.cat([self.store(store), self.sku(sku)], 1)
        return self.layers(both)

for dim, lr in [(4, 1e-3), (16, 1e-3), (4, 1e-2), (16, 1e-2)]:
    torch.manual_seed(14)
    deep = DeepTaste(dim)
    h = fit_network(deep, train_in, y_train,
                    valid_in, y_valid, lr=lr)
    print("dim %2d   lr %.3f   valid %.3f   test %.3f"
          % (dim, lr, h["valid"].min(),
             rmse_of(deep, test_in, y_test)))

dim  4   lr 0.001   valid 0.697   test 0.705


dim 16   lr 0.001   valid 0.707   test 0.711


dim  4   lr 0.010   valid 0.699   test 0.705


dim 16   lr 0.010   valid 0.539   test 0.545


### What the Vectors Learned

**Code Example 14.24** — The taste model's test errors next to the boosted model's.

In [25]:
with torch.no_grad():
    taste_pred = taste(*test_in).squeeze(1).numpy()
# Add the taste model's errors next to the boosted model's
check["taste_error"] = test["log_units"] - taste_pred
print(check.pivot_table(index="brand_tier", columns="border",
                        values=["error", "taste_error"],
                        aggfunc="mean")
      .rename(columns={False: "interior", True: "border"})
      .round(3))

                 error        taste_error       
border        interior border    interior border
brand_tier                                      
imported        -0.024  0.148      -0.027  0.005
national         0.024  0.011       0.010  0.008
private_label   -0.003  0.060      -0.003  0.035


**Code Example 14.25** — Whether the store vectors identify the border stores.

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# One row of four numbers per store, in code order
store_vec = taste.store.weight.detach().numpy()
store_info = (stores.set_index("store_id")
              .loc[pairs["store_id"].cat.categories])
is_border = store_info["city"].isin(border_cities).to_numpy()
print("border stores:", is_border.sum())

# Can the four numbers alone tell border stores apart?
auc = cross_val_score(LogisticRegression(), store_vec,
                      is_border, cv=5, scoring="roc_auc")
print("AUC from store vectors: %.3f" % auc.mean())

border stores: 55
AUC from store vectors: 1.000


**Code Example 14.26** — How far SKU vectors point along the border direction, by tier.

In [27]:
# The direction in taste space that separates border stores
fit = LogisticRegression().fit(store_vec, is_border)
direction = fit.coef_[0] / np.linalg.norm(fit.coef_[0])

sku_vec = taste.sku.weight.detach().numpy()
sku_info = (skus.set_index("sku_id")
            .loc[pairs["sku_id"].cat.categories])
# How far each SKU's vector points the same way
sku_info["border_pull"] = sku_vec @ direction
print(sku_info.groupby("brand_tier")["border_pull"]
      .agg(["mean", "std"]).round(3))

                mean    std
brand_tier                 
imported       0.358  0.493
national      -0.079  0.412
private_label -0.007  0.494


**Code Example 14.27** — The stores whose vectors point most nearly the same way as a Piedras Negras store.

In [28]:
def cosine(vectors, i):
    # Cosine similarity of row i with every row
    lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
    unit = vectors / lengths
    return unit @ unit[i]

# The first Piedras Negras store and its eight nearest stores
pn = np.flatnonzero(store_info["city"] == "Piedras Negras")[0]
nearest = np.argsort(-cosine(store_vec, pn))[1:9]
print(store_info.index[pn], store_info["city"].iloc[pn])
print(store_info.iloc[nearest][["city", "format", "nse"]])

S048 Piedras Negras
                city   format nse
S095  Piedras Negras    super   C
S105    Nuevo Laredo    super   D
S203       Monterrey    super   D
S179       Monterrey    super   D
S219        Monclova  express  C+
S051       Monterrey    super  C+
S133         Torreón    super   C
S144    Nuevo Laredo  express   C


### Embeddings as Features

**Code Example 14.28** — LightGBM with the embedding coordinates as columns.

In [29]:
def add_vectors(frame):
    # Append each pair's store and SKU vectors as columns
    out = frame.copy()
    s = frame["store_id"].cat.codes.to_numpy()
    k = frame["sku_id"].cat.codes.to_numpy()
    for j in range(4):
        out[f"store_v{j}"] = store_vec[s, j]
        out[f"sku_v{j}"] = sku_vec[k, j]
    return out

train, valid, test = (add_vectors(f)
                      for f in (train, valid, test))
vec_cols = ([f"store_v{j}" for j in range(4)]
            + [f"sku_v{j}" for j in range(4)])
hybrid = fit_boosting(attributes + vec_cols, num_leaves=15)
pred_hybrid = hybrid.predict(test[attributes + vec_cols])
print("trees %d   test %.3f"
      % (hybrid.best_iteration_,
         rmse(test["log_units"], pred_hybrid)))

trees 3298   test 0.440


---
## Inside the Language Models

### Words as Embeddings

### Attention on Mariana's Data

**Code Example 14.29** — Predicting one held-out pair as a similarity-weighted average.

In [30]:
# Unit-length store vectors, so a dot product is a cosine
unit = store_vec / np.linalg.norm(store_vec, axis=1,
                                  keepdims=True)
tr_store = train["store_id"].cat.codes.to_numpy()
tr_sku = train["sku_id"].cat.codes.to_numpy()
# What each training pair sold beyond its store and SKU levels
tr_extra = (train["log_units"]
            - levels.predict(train)).to_numpy()

def attend(store, sku, temperature):
    # Training stores that carry this SKU
    rows = np.flatnonzero(tr_sku == sku)
    similarity = unit[tr_store[rows]] @ unit[store]
    # Softmax: exponentiate, then divide by the total
    weights = np.exp(similarity / temperature)
    weights = weights / weights.sum()
    return weights @ tr_extra[rows], weights, rows

# One held-out pair: an imported SKU at store S048
te_store = test["store_id"].cat.codes.to_numpy()
te_sku = test["sku_id"].cat.codes.to_numpy()
te_levels = levels.predict(test).to_numpy()
mask = ((te_store == pn)
        & (test["brand_tier"] == "imported").to_numpy())
i = np.flatnonzero(mask)[0]
extra, w, rows = attend(te_store[i], te_sku[i], 0.3)
print(test["sku_id"].iloc[i], "actual %.2f   levels %.2f"
      "   levels + attention %.2f"
      % (test["log_units"].iloc[i], te_levels[i],
         te_levels[i] + extra))
top = np.argsort(-w)[:5]                # five largest weights
near = tr_store[rows[top]]
print(pd.DataFrame({
    "store": store_info.index[near],
    "city": store_info["city"].iloc[near].values,
    "weight": w[top].round(3),
    "extra": tr_extra[rows[top]].round(2)}))
print("stores averaged over:", len(rows))

P395 actual 3.75   levels 2.73   levels + attention 2.98
  store       city  weight  extra
0  S203  Monterrey   0.103   0.22
1  S179  Monterrey   0.101  -0.40
2  S051  Monterrey   0.085  -0.54
3  S014    Torreón   0.063   0.00
4  S007      Acuña   0.056   1.35
stores averaged over: 52


**Code Example 14.30** — The same prediction for every test pair, at four temperatures.

In [31]:
for temperature in (10, 1, 0.3, 0.1):
    preds = te_levels + np.array(
        [attend(s, k, temperature)[0]
         for s, k in zip(te_store, te_sku)])
    print("temperature %4.1f   test %.3f"
          % (temperature, rmse(test["log_units"], preds)))

temperature 10.0   test 0.688


temperature  1.0   test 0.612


temperature  0.3   test 0.514


temperature  0.1   test 0.511


### Queries, Keys and Values

### Training at Scale

---
## From Predictions to Shelves

### Scoring the Rules

**Code Example 14.31** — Every rule's score for every held-out pair.

In [32]:
# Hector's rule: rank SKUs by their average log sales in
# training stores of the same format
by_format = (train.groupby(["format", "sku_id"], observed=True)
             ["log_units"].mean())
key = list(zip(test["format"], test["sku_id"]))
hector = by_format.reindex(key).to_numpy()

with torch.no_grad():
    taste_scores = taste(*test_in).squeeze(1).numpy()
scores = pd.DataFrame({
    "store_id": test["store_id"].astype(str).to_numpy(),
    "units": test["units_per_week"].to_numpy(),
    "price": test["price_mxn"].to_numpy(),
    "hector": hector,
    "boosting": gbm.predict(test[with_ids]),
    "taste": taste_scores,
    "perfect": test["log_units"].to_numpy()})
# Keep stores with at least six held-out pairs to choose from
counts = scores.groupby("store_id")["units"].transform("size")
scores = scores[counts >= 6]
print("stores:", scores["store_id"].nunique(),
      "  candidate pairs:", len(scores))

stores: 296   candidate pairs: 6321


**Code Example 14.32** — Units sold by each rule's three picks per store.

In [33]:
def units_picked(rule, k=3):
    # Each store's k highest-scoring pairs, actual units summed
    picks = (scores.sort_values(rule, ascending=False)
             .groupby("store_id").head(k))
    return picks["units"].sum()

# A random pick of three is worth three average pairs
store_mean = scores.groupby("store_id")["units"].mean()
random_pick = (3 * store_mean).sum()
print("random      %7.0f units per week" % random_pick)
for rule in ["hector", "boosting", "taste", "perfect"]:
    print("%-11s %7.0f units per week"
          % (rule, units_picked(rule)))

random         7913 units per week
hector        17033 units per week
boosting      22499 units per week
taste         24286 units per week
perfect       27102 units per week


**Code Example 14.33** — How often the taste model and Héctor's rule pick the same SKU.

In [34]:
picked = {}
for rule in ["hector", "boosting", "taste"]:
    top = (scores.sort_values(rule, ascending=False)
           .groupby("store_id").head(3))
    picked[rule] = set(zip(top["store_id"], top.index))
same = len(picked["taste"] & picked["hector"])
print("picks shared by the taste model and Hector: %d of %d"
      % (same, len(picked["taste"])))

picks shared by the taste model and Hector: 418 of 888


**Code Example 14.34** — Peso sales of the picks over one season.

In [35]:
# Peso sales of the picks over a 26-week season
scores["pesos"] = scores["units"] * scores["price"]
def pesos_picked(rule, k=3):
    picks = (scores.sort_values(rule, ascending=False)
             .groupby("store_id").head(k))
    return picks["pesos"].sum() * 26
base = pesos_picked("hector")
for rule in ["hector", "boosting", "taste"]:
    season = pesos_picked(rule)
    print("%-9s %12s pesos   vs Hector %+5.1f%%"
          % (rule, f"{season:,.0f}", 100 * (season / base - 1)))

hector      15,060,931 pesos   vs Hector  +0.0%
boosting    20,572,557 pesos   vs Hector +36.6%
taste       23,146,051 pesos   vs Hector +53.7%


### The Recommendations

**Code Example 14.35** — Three new SKUs for every store.

In [36]:
# Every pair a store does NOT carry yet
grid = pd.MultiIndex.from_product(
    [range(n_stores), range(n_skus)], names=["s", "k"])
carried = set(zip(pairs["store_id"].cat.codes,
                  pairs["sku_id"].cat.codes))
new = pd.DataFrame(index=grid).reset_index()
new = new[[(s, k) not in carried
           for s, k in zip(new["s"], new["k"])]]
with torch.no_grad():
    new["score"] = taste(torch.tensor(new["s"].to_numpy()),
                         torch.tensor(new["k"].to_numpy())
                         ).squeeze(1).numpy()
# Three highest-scoring new SKUs for each store
recs = (new.sort_values("score", ascending=False)
        .groupby("s").head(3))
recs["store_id"] = store_info.index[recs["s"]]
recs = recs.join(sku_info[["category", "brand_tier"]]
                 .reset_index(drop=True), on="k")
recs["sku_id"] = sku_info.index[recs["k"]]
recs["border"] = is_border[recs["s"]]
print("uncarried pairs scored:", len(new),
      "  recommendations:", len(recs))
print(recs[recs["store_id"] == "S048"]
      [["store_id", "sku_id", "category", "brand_tier"]])
share = (recs.assign(imported=recs["brand_tier"] == "imported")
         .groupby("border")["imported"].mean())
print("share imported, interior %.2f   border %.2f"
      % (share[False], share[True]))

uncarried pairs scored: 88305   recommendations: 900
      store_id sku_id category     brand_tier
19255     S048   P055    dairy  private_label
19234     S048   P034    dairy       imported
19430     S048   P230    dairy       imported
share imported, interior 0.14   border 0.34


### Products With No History

**Code Example 14.36** — Forty SKUs treated as new, and the boosted model without the SKU identifier.

In [37]:
# Forty SKUs treated as new: none of their pairs is seen
rng_new = np.random.default_rng(140)
new_skus = rng_new.choice(pairs["sku_id"].cat.categories, 40,
                          replace=False)
is_new = pairs["sku_id"].isin(new_skus)
launch = pairs[is_new]                  # all their pairs
old_train = train[~train["sku_id"].isin(new_skus)]
old_valid = valid[~valid["sku_id"].isin(new_skus)]
print("pairs of new SKUs:", len(launch),
      "  training pairs left:", len(old_train))

# Boosting without the SKU identifier, which a new SKU lacks
cold_cols = attributes + ["store_id"]
model = lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.03,
                          num_leaves=15, verbose=-1)
model.fit(old_train[cold_cols], old_train["log_units"],
          eval_set=[(old_valid[cold_cols],
                     old_valid["log_units"])],
          callbacks=[lgb.early_stopping(100, verbose=False)])
y_launch = launch["log_units"]
print("mean          %.3f"
      % rmse(y_launch, old_train["log_units"].mean()))
print("boosting      %.3f"
      % rmse(y_launch, model.predict(launch[cold_cols])))

pairs of new SKUs: 3216   training pairs left: 19914


mean          0.895


boosting      0.844


**Code Example 14.37** — Store vectors learned as before, SKU vectors computed from attributes.

In [38]:
sku_cols = ["category", "brand_tier", "pack"]
def sku_design(frame):
    # A SKU described only by its attributes, never its name
    dummies = pd.get_dummies(frame[sku_cols], dtype=float)
    price = (frame[["log_price"]] - center["log_price"]
             ) / scale["log_price"]
    return to_tensor(pd.concat([dummies, price], axis=1))

class AttributeTaste(nn.Module):
    # Store vectors as before; SKU vector and level computed
    # from the SKU's attributes by a linear layer
    def __init__(self, dim, n_attr):
        super().__init__()
        self.store = nn.Embedding(n_stores, dim)
        self.store_level = nn.Embedding(n_stores, 1)
        nn.init.normal_(self.store.weight, 0, 0.1)
        nn.init.zeros_(self.store_level.weight)
        self.sku = nn.Linear(n_attr, dim + 1)  # vector + level
        self.mean = nn.Parameter(y_train.mean().reshape(1))

    def forward(self, store, attrs):
        out = self.sku(attrs)
        level, vec = out[:, :1], out[:, 1:]
        match = (self.store(store) * vec).sum(1, keepdim=True)
        return (self.mean + self.store_level(store)
                + level + match)

tr_in = (codes(old_train, "store_id"), sku_design(old_train))
va_in = (codes(old_valid, "store_id"), sku_design(old_valid))
la_in = (codes(launch, "store_id"), sku_design(launch))
torch.manual_seed(14)
cold = AttributeTaste(4, tr_in[1].shape[1])
fit_network(cold, tr_in, to_tensor(old_train["log_units"]),
            va_in, to_tensor(old_valid["log_units"]))
print("attribute taste %.3f"
      % rmse_of(cold, la_in, to_tensor(y_launch)))
# For comparison: the full taste model on the test pairs of
# the same 40 SKUs, which it saw in training through other pairs
in_test = test["sku_id"].isin(new_skus).to_numpy()
print("test pairs of these SKUs:", in_test.sum(),
      "  taste model with history %.3f"
      % rmse(test["log_units"].to_numpy()[in_test],
             taste_scores[in_test]))

attribute taste 0.761
test pairs of these SKUs: 638   taste model with history 0.446


---
## Exercises

In [39]:
# Exercise 14.1: One run is one draw
# Retrain the four-dimensional taste model of Code Example 
# 14.22 with five different seeds and report the spread of its 
# test errors. Do the same for LightGBM with the identifiers, 
# varying `random\_state` and the `subsample` options. Is the 
# difference between the two models larger than the spread of 
# either, and what would you tell Héctor if it were not?

In [40]:
# Exercise 14.2: Regularizing the vectors
# The embeddings with eight and sixteen coordinates overfit 
# and stopped early. Replace Adam with `torch.optim.AdamW`, 
# which shrinks every weight a little toward zero at each 
# step, and retrain the four sizes with `weight\_decay` values 
# of 0.01 and 0.1. Does a larger embedding now beat four 
# coordinates on the validation set? Explain why shrinking the 
# weights makes extra coordinates less harmful.

In [41]:
# Exercise 14.3: Clustering stores by taste
# Apply the $k$-means method of the chapter to the 300 store 
# vectors, for $k$ between three and six, after scaling each 
# vector to length one so that Euclidean distances rank pairs 
# of stores as cosine similarity does. Compare the clusters 
# with the stores' formats, cities and neighborhood levels. 
# Which clusters can you describe from the store table, and 
# which can you not? Write a one-paragraph description of the 
# cluster that is hardest to explain, as you would for a 
# category manager.

In [42]:
# Exercise 14.4: A better rule without a model
# Modify Héctor's rule in Code Example 14.31 so that a SKU's 
# score is its average log sales in training stores of the 
# same format and the same location, border or interior. 
# Report the share of the attainable gain the new rule covers. 
# How much of the gap between Héctor's rule and the taste 
# model does one known interaction close, and what does the 
# remainder represent?

In [43]:
# Exercise 14.5: What a new product takes away
# The evaluation of the chapter counts the sales of each new 
# SKU and nothing else, but a new SKU on the shelf can take 
# sales from products the store already carries. Explain why 
# this effect, called *cannibalization*, is likely to be 
# larger for some of the taste model's picks than for others, 
# and propose a measurement that the pilot of the warning box 
# in the chapter could add to capture it.

In [44]:
# Exercise 14.6: A memo to Héctor
# (open-ended). Write a one-page recommendation for the spring 
# reset that a commercial director with no background in 
# machine learning can act on. It should say what the taste 
# model is in two sentences, what it would gain and how sure 
# Mariana can be, what could make the estimate wrong, and how 
# the pilot would settle the question within a season.